# Qa attention weights

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA: Cross-Attention Weights

This notebook inspects cross-attention matrices from the coastal-conditioned transformer.

Outputs are saved under: `results/<run>/attention_weight_diagnostics/`

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")

from src.data_pipeline import load_point_centric_arrays, build_split_dataloader
from src.models import build_model_from_config
from notebooks.multisource_notebook_helpers import resolve_point_centric_dir

CFG_PATH = ROOT / "configs" / "training.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text())
POINT_CENTRIC_DIR = resolve_point_centric_dir(CFG_PATH)

# Controls
SPLIT = "test"  # 'val' or 'test'
MAX_BATCHES = 40  # Limit for quick QA
STORM_QUANTILE = 0.80  # Storm threshold in Hs percentile
SELECT_TASK_FOR_HEAD_PLOT = "dir"

results_dir = ROOT / cfg.get("logging", {}).get("output_dir", "results/coastal_transformer")
ckpt_name = cfg.get("logging", {}).get("checkpoint_name", "coastal_transformer_best.pt")
ckpt_path = results_dir / ckpt_name
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

out_dir = results_dir / "attention_weight_diagnostics"
out_dir.mkdir(parents=True, exist_ok=True)

arrays = load_point_centric_arrays(str(POINT_CENTRIC_DIR))
loader, ds = build_split_dataloader(arrays, cfg, split_name=SPLIT, shuffle=False)

sample = ds[0]
model = build_model_from_config(
    cfg,
    dynamic_input_dim=int(sample["x_dynamic"].shape[-1]),
    static_input_dim=int(sample["x_static"].shape[-1]),
    output_dim=int(sample["y"].shape[-1]),
)

checkpoint = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

output_columns = cfg.get("data", {}).get(
    "output_columns", ["hs", "tp", "dir_sin", "dir_cos", "dp_sin", "dp_cos"]
)
task_labels = ["hs", "tp", "dir", "dp"]
hs_idx = output_columns.index("hs") if "hs" in output_columns else 0

meta_path = POINT_CENTRIC_DIR / "point_centric_metadata.json"
hs_mean = 0.0
hs_scale = 1.0
if meta_path.exists():
    payload = json.loads(meta_path.read_text())
    scaler = (payload.get("normalization") or {}).get("target_scaler") or {}
    f_names = list(scaler.get("feature_names", []) or [])
    means = list(scaler.get("mean", []) or [])
    scales = list(scaler.get("scale", []) or [])
    if len(f_names) == len(means) == len(scales):
        lut = {str(n): i for i, n in enumerate(f_names)}
        hs_name = str(output_columns[hs_idx])
        if hs_name in lut:
            j = lut[hs_name]
            hs_mean = float(means[j])
            hs_scale = float(scales[j]) if float(scales[j]) != 0.0 else 1.0

print("Split:", SPLIT)
print("Checkpoint:", ckpt_path)
print("Output dir:", out_dir)
print("Point-centric dir:", POINT_CENTRIC_DIR)

In [ ]:
all_attn = []
all_hs = []

with torch.no_grad():
    for batch_idx, batch in enumerate(loader):
        if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
            break

        x_dyn = batch["x_dynamic"]
        x_stat = batch["x_static"]
        y = batch["y"]

        out = model(x_dyn, x_stat, return_attention=True)
        attn = out["cross_attention_weights"]  # [B, heads, task_queries, context_tokens]

        if attn is None:
            raise RuntimeError(
                "No attention weights returned. Ensure model forward supports return_attention=True."
            )

        hs_scaled = y[:, hs_idx].cpu().numpy()
        hs_true = (hs_scaled * hs_scale) + hs_mean

        all_attn.append(attn.cpu().numpy())
        all_hs.append(hs_true)

if not all_attn:
    raise RuntimeError("No attention batches collected. Increase MAX_BATCHES or check loader.")

attn_all = np.concatenate(all_attn, axis=0)  # [N, H, Q, K]
hs_all = np.concatenate(all_hs, axis=0)

N, H, Q, K = attn_all.shape
print("Attention tensor shape:", attn_all.shape)

mean_task_token = attn_all.mean(axis=(0, 1))  # [Q, K]

task_to_idx = {name: i for i, name in enumerate(task_labels)}
selected_task_idx = task_to_idx.get(SELECT_TASK_FOR_HEAD_PLOT, 2)
per_head_selected = attn_all[:, :, selected_task_idx, :].mean(axis=0)  # [H, K]

storm_thr = np.quantile(hs_all, STORM_QUANTILE)
storm_mask = hs_all >= storm_thr
calm_mask = ~storm_mask

storm_mean = (
    attn_all[storm_mask].mean(axis=(0, 1)) if np.any(storm_mask) else np.full((Q, K), np.nan)
)
calm_mean = attn_all[calm_mask].mean(axis=(0, 1)) if np.any(calm_mask) else np.full((Q, K), np.nan)
diff_mean = storm_mean - calm_mean

token_labels = [f"t-{K - 2 - i}" for i in range(K - 1)] + ["static"]

fig1, ax1 = plt.subplots(figsize=(12, 4), dpi=120)
im1 = ax1.imshow(mean_task_token, aspect="auto", cmap="viridis")
ax1.set_title(f"Cross-Attention Mean by Task ({SPLIT})")
ax1.set_yticks(np.arange(Q))
ax1.set_yticklabels(task_labels)
ax1.set_xticks(np.arange(K))
ax1.set_xticklabels(token_labels, rotation=45, ha="right")
fig1.colorbar(im1, ax=ax1, label="attention weight")
fig1.tight_layout()
p1 = out_dir / f"attention_mean_task_token_{SPLIT}.png"
fig1.savefig(p1, dpi=200, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 4), dpi=120)
im2 = ax2.imshow(per_head_selected, aspect="auto", cmap="magma")
ax2.set_title(f"Per-Head Attention for Task={task_labels[selected_task_idx]} ({SPLIT})")
ax2.set_yticks(np.arange(H))
ax2.set_yticklabels([f"head_{i}" for i in range(H)])
ax2.set_xticks(np.arange(K))
ax2.set_xticklabels(token_labels, rotation=45, ha="right")
fig2.colorbar(im2, ax=ax2, label="attention weight")
fig2.tight_layout()
p2 = out_dir / f"attention_per_head_{task_labels[selected_task_idx]}_{SPLIT}.png"
fig2.savefig(p2, dpi=200, bbox_inches="tight")
plt.show()

fig3, axs = plt.subplots(1, 3, figsize=(18, 4), dpi=120, sharey=True)
im3a = axs[0].imshow(calm_mean, aspect="auto", cmap="viridis")
axs[0].set_title(f"Calm Mean (Hs < Q{int(STORM_QUANTILE * 100)})")
im3b = axs[1].imshow(storm_mean, aspect="auto", cmap="viridis")
axs[1].set_title(f"Storm Mean (Hs >= Q{int(STORM_QUANTILE * 100)})")
im3c = axs[2].imshow(diff_mean, aspect="auto", cmap="coolwarm")
axs[2].set_title("Storm - Calm")
for ax in axs:
    ax.set_yticks(np.arange(Q))
    ax.set_yticklabels(task_labels)
    ax.set_xticks(np.arange(K))
    ax.set_xticklabels(token_labels, rotation=45, ha="right")
fig3.colorbar(im3c, ax=axs.ravel().tolist(), label="attention weight")
fig3.tight_layout()
p3 = out_dir / f"attention_calm_vs_storm_{SPLIT}.png"
fig3.savefig(p3, dpi=200, bbox_inches="tight")
plt.show()

rows = []
for q_idx, task_name in enumerate(task_labels):
    for k_idx in range(K):
        rows.append(
            {
                "split": SPLIT,
                "task": task_name,
                "head": "all",
                "token_index": int(k_idx),
                "token_label": token_labels[k_idx],
                "attention_mean": float(mean_task_token[q_idx, k_idx]),
                "calm_mean": float(calm_mean[q_idx, k_idx])
                if np.isfinite(calm_mean[q_idx, k_idx])
                else np.nan,
                "storm_mean": float(storm_mean[q_idx, k_idx])
                if np.isfinite(storm_mean[q_idx, k_idx])
                else np.nan,
                "storm_minus_calm": float(diff_mean[q_idx, k_idx])
                if np.isfinite(diff_mean[q_idx, k_idx])
                else np.nan,
            }
        )

for head_idx in range(H):
    for q_idx, task_name in enumerate(task_labels):
        vec = attn_all[:, head_idx, q_idx, :].mean(axis=0)
        for k_idx in range(K):
            rows.append(
                {
                    "split": SPLIT,
                    "task": task_name,
                    "head": f"head_{head_idx}",
                    "token_index": int(k_idx),
                    "token_label": token_labels[k_idx],
                    "attention_mean": float(vec[k_idx]),
                    "calm_mean": np.nan,
                    "storm_mean": np.nan,
                    "storm_minus_calm": np.nan,
                }
            )

summary_df = pd.DataFrame(rows)
csv_path = out_dir / f"attention_summary_{SPLIT}.csv"
summary_df.to_csv(csv_path, index=False)

print("Saved:")
print(" -", p1)
print(" -", p2)
print(" -", p3)
print(" -", csv_path)
summary_df.head()